# Session 10: An exact robust route search in C++

**Question:** can the planner find the route that is best in the *worst* of several weather scenarios, exactly and fast enough to run on every night?

**Builds on:** Session 9's planner finds the cheapest path for one weather field. Sessions 7–8 showed that errors in the height of the moist layer change route choices on most nights, so the route should be chosen against several shifted fields at once.

**Approach:**
1. Seven scenarios: the weather field as given, and moved up or down by 150, 300 and 600 m (as Sessions 7–8). Each gives a cost model on the same grid: the same geometry and fuel, a different contrail field.
2. Two rules, as before: **worst case** (minimise the largest J = fuel + λ·PCR km over the scenarios) and **minimum regret** (minimise the largest gap to the best possible path in each scenario).
3. Neither rule splits into a sum over stages, so Session 9's dynamic programming cannot be used. The search keeps, at each grid point, every partial path not beaten on fuel and on PCR km in all seven scenarios, and drops partial paths that cannot beat the best complete path known, using lower bounds on the cost still to come. This is exact on the grid.
4. The search is written in C++ (`cpp/src/robust_search.cpp`, called from Python through pybind11) with an identical pure-Python version for testing.

**Checks:** both versions match brute force over every possible path on small grids, return the same path step for step, and reduce to the Session 9 planner when there is one scenario (automated tests). On real data, this notebook checks that they agree again and that the search stays fast.

**Assumptions:** as Session 9, with flight levels FL340–FL390 only: moving the weather field 600 m up or down leaves FL310–FL330 outside the data (as in Sessions 7–8). This session uses great-circle timing and does not re-fly paths; both come in Session 11.

**Sections:** 1. Setup · 2. Scenario cost models for one night · 3. C++ and Python agree on real data · 4. Speed and size of the search · 5. Robust vs ordinary paths · Results

## 1. Setup

The C++ core must be built once with `python -m pip install --no-build-isolation ./cpp`; `contrail_core` below must not be `None`.

In [1]:
import sys, time, warnings
from dataclasses import replace
from pathlib import Path
SRC = (Path("..") / "src").resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from contrail_routing import config
from contrail_routing.met import load_era5_checked, load_region
from contrail_routing.geometry import haversine_km
from contrail_routing.scenarios import shift_field
from contrail_routing.planner import build_cost_model, plan, path_costs, climb_kg_per_level
from contrail_routing.robust import robust_plan, contrail_core

warnings.filterwarnings("ignore", message=r"\s*Met data appears to have originated from ECMWF")
pd.set_option("display.width", 220)
print("C++ core:", contrail_core)
assert contrail_core is not None, "build it first: python -m pip install --no-build-isolation ./cpp"

results_dir = Path("..") / "results"
fig_dir = results_dir / "figures"
s8_dir, s10_dir = results_dir / "session8", results_dir / "session10"
s10_dir.mkdir(parents=True, exist_ok=True)

DATE = "2022-03-01"
N_STAGES = 60
OFFSETS_KM = np.arange(-300, 301, 25)
FLIGHT_LEVELS = np.array([340, 350, 360, 370, 380, 390])
VERTICAL_M = {"base": 0.0, "up150": 150.0, "up300": 300.0, "up600": 600.0,
              "down150": -150.0, "down300": -300.0, "down600": -600.0}
MASSES = {"light_56t": 56000.0, "model_default": None,
          "heavy_68t": 68000.0, "very_heavy_72t": 72000.0}
LAMBDAS = [0.1, 1.0, 10.0]
RULES = ["worst_case", "min_regret"]
GC_KM = float(haversine_km(*config.ORIGIN, *config.DESTINATION))
fuel8 = pd.read_csv(s8_dir / "fuel_by_night.csv")

def fuel_rates(date, mass_case):
    """kg/km per flight level (inf outside the envelope) and start mass, from the Session 8 great circle."""
    g = fuel8[(fuel8["date"] == date) & (fuel8["mass_case"] == mass_case)].set_index("route")
    gc_rows = g.loc[[f"GC0_FL{fl}" for fl in FLIGHT_LEVELS]]
    rate = np.where(gc_rows["feasible"], gc_rows["fuel_kg"] / GC_KM, np.inf)
    return rate, float(gc_rows.loc[gc_rows["feasible"], "start_mass_kg"].mean())

C++ core: <module 'contrail_core' from 'C:\\Users\\barqb\\robust-contrail-routing\\.venv\\Lib\\site-packages\\contrail_core.cp313-win_amd64.pyd'>


## 2. Scenario cost models for one night

ERA5 for 2022-03-01 is shifted up and down as in Sessions 7–8, and a cost model is built for each shifted field. The contrail fields do not depend on the aircraft, so they are built once; fuel rates are swapped in per mass.

In [2]:
met, _ = load_era5_checked((f"{DATE}T00", f"{DATE}T06"), config.LEVELS_FINE, config.REGION_LON, config.REGION_LAT)
region = load_region(met, config.REGION_LON, config.REGION_LAT)
met.data.close()

rate0, mass0 = fuel_rates(DATE, "model_default")
t0 = time.time()
base_models = {name: build_cost_model(shift_field(region, up_m=dz), config.ORIGIN, config.DESTINATION,
                                      pd.Timestamp(f"{DATE} 00:00"), config.CRUISE_SPEED_MS, N_STAGES,
                                      OFFSETS_KM, FLIGHT_LEVELS, rate0, climb_kg_per_level(mass0))
               for name, dz in VERTICAL_M.items()}
print(f"{len(base_models)} scenario cost models in {time.time() - t0:.1f} s")
share = pd.DataFrame({name: m.pcr_frac[:, :, m.max_dj, :, m.max_dk].mean(axis=(0, 1))
                      for name, m in base_models.items()}, index=[f"FL{fl}" for fl in FLIGHT_LEVELS])
print("\nShare of straight-and-level steps in PCR conditions, by level and scenario:")
print(share.round(3).to_string())

def models_for(mass_case):
    rate, m_est = fuel_rates(DATE, mass_case)
    climb = climb_kg_per_level(MASSES[mass_case] or m_est)
    return [replace(m, fuel_per_km=rate, climb_kg=climb) for m in base_models.values()]

ERA5 [175, 190, 200, 210, 220, 230, 240, 250, 275, 300] hPa: all 7 hours complete.
7 scenario cost models in 2.1 s

Share of straight-and-level steps in PCR conditions, by level and scenario:
        base  up150  up300  up600  down150  down300  down600
FL340  0.355  0.484  0.504  0.412    0.366    0.344    0.229
FL350  0.344  0.366  0.355  0.504    0.318    0.229    0.108
FL360  0.227  0.316  0.343  0.355    0.196    0.105    0.001
FL370  0.102  0.196  0.224  0.343    0.042    0.001    0.000
FL380  0.000  0.042  0.098  0.223    0.000    0.000    0.000
FL390  0.000  0.000  0.000  0.094    0.000    0.000    0.000


## 3. C++ and Python agree on real data

The automated tests compare the two versions on random grids; here they are compared once on the real contrail fields (model-estimate mass, λ = 1, both rules). The pure-Python version takes a few seconds to a minute.

In [3]:
models0 = models_for("model_default")
rows = []
for rule in RULES:
    out = {}
    for engine in ["cpp", "python"]:
        t0 = time.time()
        out[engine] = robust_plan(models0, 1.0, rule=rule, engine=engine)
        out[engine]["seconds"] = time.time() - t0
    same = (np.array_equal(out["cpp"]["j_idx"], out["python"]["j_idx"])
            and np.array_equal(out["cpp"]["k_idx"], out["python"]["k_idx"])
            and out["cpp"]["labels_created"] == out["python"]["labels_created"])
    rows.append({"rule": rule, "same_path_and_search": same,
                 "objective_cpp": out["cpp"]["objective"], "objective_python": out["python"]["objective"],
                 "labels": out["cpp"]["labels_created"],
                 "seconds_cpp": out["cpp"]["seconds"], "seconds_python": out["python"]["seconds"]})
agree = pd.DataFrame(rows)
agree["speed_up"] = agree["seconds_python"] / agree["seconds_cpp"]
print(agree.round(3).to_string(index=False))
assert agree["same_path_and_search"].all()
print("\nC++ and Python give the same path, objective and search on real data.")

      rule  same_path_and_search  objective_cpp  objective_python  labels  seconds_cpp  seconds_python  speed_up
worst_case                  True       4923.956          4923.956      98        0.010           0.838    83.126
min_regret                  True          1.977             1.977      98        0.431           1.270     2.947

C++ and Python give the same path, objective and search on real data.


## 4. Speed and size of the search

Every mass × λ × rule, for a planner holding one flight level and one free to change level. "Labels" counts the partial paths the search created; "pruned" those dropped by the bound. The minimum-regret rule first runs the Session 9 planner once per scenario to find each scenario's best cost; that time is included.

In [4]:
rows, paths = [], []
for mc in MASSES:
    models = models_for(mc)
    for lam in LAMBDAS:
        for rule in RULES:
            for variant, max_dk in {"constant_level": 0, "free_level": 1}.items():
                t0 = time.time()
                r = robust_plan(models, lam, rule=rule, max_dk=max_dk, engine="cpp")
                rows.append({"mass_case": mc, "lambda": lam, "rule": rule, "variant": variant,
                             "seconds": time.time() - t0, "labels": r["labels_created"],
                             "pruned": r["labels_pruned_by_bound"], "max_per_point": r["max_labels_at_a_point"],
                             "objective": r["objective"], "fuel_kg": r["fuel_kg"],
                             "pcr_worst_km": float(r["pcr_km"].max()), "pcr_base_km": float(r["pcr_km"][0]),
                             "levels": "/".join(map(str, sorted(set(r["flight_level"].tolist())))),
                             "max_offset_km": float(np.abs(r["offset_km"]).max())})
                paths.append(pd.DataFrame({"mass_case": mc, "lambda": lam, "rule": rule, "variant": variant,
                                           "stage": np.arange(len(r["offset_km"])),
                                           "offset_km": r["offset_km"], "flight_level": r["flight_level"]}))
runs = pd.DataFrame(rows)
print(runs.drop(columns=["objective"]).round(1).to_string(index=False))
print(f"\n{len(runs)} searches: total {runs['seconds'].sum():.1f} s, slowest {runs['seconds'].max():.2f} s, "
      f"most labels {runs['labels'].max():,}")
runs.to_csv(s10_dir / "robust_search_runs_2022-03-01.csv", index=False)
pd.concat(paths, ignore_index=True).to_csv(s10_dir / "robust_paths_2022-03-01.csv", index=False)

     mass_case  lambda       rule        variant  seconds  labels  pruned  max_per_point  fuel_kg  pcr_worst_km  pcr_base_km              levels  max_offset_km
     light_56t     0.1 worst_case constant_level      0.0      66     133              1   4387.2           0.0          0.0                 390            0.0
     light_56t     0.1 worst_case     free_level      0.0      66     337              1   4387.2           0.0          0.0                 390            0.0
     light_56t     0.1 min_regret constant_level      0.1      66     133              1   4387.2           0.0          0.0                 390            0.0
     light_56t     0.1 min_regret     free_level      0.4      66     337              1   4387.2           0.0          0.0                 390            0.0
     light_56t     1.0 worst_case constant_level      0.0      66     133              1   4387.2           0.0          0.0                 390            0.0
     light_56t     1.0 worst_case     fr

## 5. Robust vs ordinary paths

The ordinary planner (Session 9) optimised for the unshifted field only, evaluated in all seven scenarios with the same cost model, against the robust path. The robust path can only be equal or better in its worst scenario; the question is what it costs in fuel and in the unshifted case.

In [5]:
rows = []
for mc in MASSES:
    models = models_for(mc)
    for lam in LAMBDAS:
        p = plan(models[0], lam, max_dk=1)                     # ordinary plan, unshifted field
        J_ord = []
        for m in models:
            c = path_costs(m, p["j_idx"], p["k_idx"])
            J_ord.append(c["fuel_kg"] + lam * c["pcr_km"])
        r = robust_plan(models, lam, rule="worst_case", max_dk=1, engine="cpp")
        rows.append({"mass_case": mc, "lambda": lam,
                     "ordinary_levels": "/".join(map(str, sorted(set(p["flight_level"].tolist())))),
                     "robust_levels": "/".join(map(str, sorted(set(r["flight_level"].tolist())))),
                     "ordinary_worst_J": max(J_ord), "robust_worst_J": r["objective"],
                     "worst_case_saving": max(J_ord) - r["objective"],
                     "extra_fuel_kg": r["fuel_kg"] - p["fuel_kg"],
                     "ordinary_pcr_worst_km": max(J - p["fuel_kg"] for J in J_ord) / lam,
                     "robust_pcr_worst_km": float(r["pcr_km"].max())})
cmp_ = pd.DataFrame(rows)
print(cmp_.round(1).to_string(index=False))
assert (cmp_["worst_case_saving"] >= -1e-6).all(), "robust path worse in its worst case than the ordinary one"
cmp_.to_csv(s10_dir / "robust_vs_ordinary_2022-03-01.csv", index=False)

     mass_case  lambda ordinary_levels       robust_levels  ordinary_worst_J  robust_worst_J  worst_case_saving  extra_fuel_kg  ordinary_pcr_worst_km  robust_pcr_worst_km
     light_56t     0.1             390                 390            4387.2          4387.2                0.0           -0.0                    0.0                  0.0
     light_56t     1.0             390                 390            4387.2          4387.2                0.0           -0.0                    0.0                  0.0
     light_56t    10.0             390                 390            4387.2          4387.2                0.0           -0.0                    0.0                  0.0
 model_default     0.1             380             380/390            4963.0          4924.0               39.0            2.0                  410.0                  0.0
 model_default     1.0             380             380/390            5332.0          4924.0              408.0            2.0                  4

## Results

### Setup

The robust search was run on 2022-03-01 with seven scenarios: the ERA5 field as given and moved up or down by 150, 300 and 600 m. Grid: 61 stage boundaries × 25 lateral offsets (−300 to +300 km) × 6 flight levels (FL340–FL390), fuel rates from the Session 8 great circle for each mass, great-circle timing. Four masses, λ = 0.1, 1 and 10 kg/km, both rules (worst case, minimum regret), for a planner holding one flight level and one free to change level: 48 searches.

### Checks

- On the real contrail fields, the C++ core and the pure-Python version returned the same path, the same objective (worst case 4,923.956; minimum regret 1.977) and the same number of partial paths at every step.
- Automated tests (78 in total): both versions match brute force over every possible path on small grids for both rules; they return identical paths and search counts on random grids; with one scenario the objective equals the Session 9 planner's; identical scenarios give the single-scenario answer and zero regret.
- In every case the robust path's worst-case cost is at most the ordinary plan's (asserted).

### Findings

#### Speed

| | C++ | Python | Speed-up |
|---|---|---|---|
| Worst case, ≈64 t, λ = 1 (98 partial paths) | 0.010 s | 0.84 s | 83× |
| Minimum regret, same case | 0.43 s | 1.27 s | 2.9× (both dominated by the seven single-scenario plans run first) |

The 48 searches took 126 s in total. 45 took 1 s or less; the three slow ones were all the 72 t aircraft with free level changes (15.5, 19.5 and 84 s). The slowest created 3.0 million partial paths, up to 11,837 at one grid point: real contrail fields near the cruise levels of a heavy aircraft leave many partial paths that are each best in some scenario.

#### Robust vs ordinary paths (worst case, free level, planner estimates)

| Mass | Ordinary plan (unshifted field) | Its worst scenario | Robust plan | Its worst scenario | Extra fuel |
|---|---|---|---|---|---|
| 56 t | FL390 | 0 km | identical | 0 km | 0 kg |
| ≈64 t | FL380 | 410 km | FL380 → FL390 | 0 km | +2 kg |
| 68 t | FL370 | 980 km | FL370 → FL390 | 0 km at λ ≥ 1 (220 km at λ = 0.1) | +39 kg (+12 kg at λ = 0.1) |
| 72 t | FL350–FL370 | 1,030–1,180 km | FL340–FL380, up to 100 km sideways | 530 / 168 / 39 km at λ = 0.1 / 1 / 10 | +42 / +111 / +255 kg |

For the lighter aircraft a step climb to a level above the moist layer in every scenario removes the risk for a few kg of fuel. The 68 t robust plan reaches zero exposure in every scenario for 39 kg, by climbing in steps; the best fixed route in Session 7 needed 50 kg at FL390. A 72 t aircraft cannot climb high enough, and its robust plans trade fuel for a smaller worst case.

### Interpretation

The exact robust search is practical: most cases take well under a second, and the C++ core makes the hardest ones (minutes in C++, an estimated hours in Python) feasible. The robust paths behave as expected physically, climbing above the moist layer when the aircraft can and bending around it when it cannot. These are the planner's own estimates on one night; Session 11 re-flies the robust paths through the full pipeline on all 24 nights.

### Corrections

- A first build of the C++ core could not return lists to Python (a missing pybind11 header); the automated tests caught it before any results were produced.
- The robust grid uses FL340–FL390 only: moving the weather field 600 m up or down leaves FL310–FL330 outside the data, as in Sessions 7–8.

### Limitations

- One night, great-circle timing, and no re-flying of robust paths through the full pipeline (Session 11).
- The search's cost grows with the number of partial paths that are each best in some scenario; for heavy aircraft on moist nights this reached 3 million. A tighter starting bound and a memory limit suited to an 8 GB laptop are needed before running all 24 nights.
- Scenario sizes (150–600 m) are sensitivity cases, weighted equally, not calibrated estimates of ERA5 error.